# 04 — Model Training

This notebook trains the `SentimentLSTM` architecture defined in `03_model_building.ipynb`
using a single, carefully tuned configuration.

**Prerequisites:** `02_preprocessing.ipynb` must have been run first so that the following files exist:
- `data/vocab.pkl`
- `data/train_sequences.npy`, `data/train_labels.npy`
- `data/val_sequences.npy`, `data/val_labels.npy`
- `data/test_sequences.npy`, `data/test_labels.npy`

**Notebook outline:**
1. Setup — imports, device detection, reproducibility seed
2. Dataset and DataLoaders
3. EarlyStopping and helper functions
4. Plotting helper
5. Model configuration and training
6. Results — training curves and interpretation

## Setup

Add the repository root to `sys.path` so that `src/` is importable both locally and in Google Colab.
We also set the random seed across Python, NumPy, and PyTorch to ensure reproducible weight
initialisation and data shuffling.

In [1]:
import sys, os

# ── Colab vs. local ortam tespiti ────────────────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.preprocessing import Vocabulary
from src.model import SentimentLSTM

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Working directory: /Users/azrakarakaya/IMDb_Sentiment_Analysis
Using device: cpu


## Section 1 — Load Data

We reload the pre-computed artefacts written by `02_preprocessing.ipynb`:
- `data/vocab.pkl` — the `Vocabulary` object, used here only to get `vocab_size` for the embedding layer
- The six `.npy` files — integer sequences (shape `(N, 500)`) and float labels (shape `(N,)`)

Loading from `.npy` files is much faster than re-running the preprocessing pipeline
(seconds vs. minutes), which is important when iterating on training hyperparameters.

In [ ]:
# ── Preprocessed dosyalar yoksa Drive'dan kopyala ───────────────────────
drive_dir = '/content/drive/MyDrive/IMDb_preprocessed'
required = ['vocab.pkl', 'train_sequences.npy', 'train_labels.npy',
            'val_sequences.npy', 'val_labels.npy',
            'test_sequences.npy', 'test_labels.npy']

if not os.path.exists('data/vocab.pkl') and os.path.exists(drive_dir):
    import shutil
    os.makedirs('data', exist_ok=True)
    for fname in required:
        shutil.copy(f'{drive_dir}/{fname}', f'data/{fname}')
    print('Preprocessed dosyalar Drive\'dan yüklendi.')
elif os.path.exists('data/vocab.pkl'):
    print('Preprocessed dosyalar zaten mevcut.')
else:
    print('UYARI: Drive\'da da dosya yok. Önce 02_preprocessing.ipynb çalıştır.')

In [2]:
# ── Verify prerequisite files exist ──────────────────────────────────────
required_files = [
    "data/vocab.pkl",
    "data/train_sequences.npy", "data/train_labels.npy",
    "data/val_sequences.npy",   "data/val_labels.npy",
    "data/test_sequences.npy",  "data/test_labels.npy",
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing prerequisite files: {missing}\n"
        "Please run 02_preprocessing.ipynb first to generate these files."
    )

# ── Load vocabulary ───────────────────────────────────────────────────────
vocab = Vocabulary.load("data/vocab.pkl")
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE:,}")

# ── Load encoded sequences and labels ────────────────────────────────────
train_sequences = np.load("data/train_sequences.npy")
train_labels    = np.load("data/train_labels.npy")
val_sequences   = np.load("data/val_sequences.npy")
val_labels      = np.load("data/val_labels.npy")
test_sequences  = np.load("data/test_sequences.npy")
test_labels     = np.load("data/test_labels.npy")

print(f"\nLoaded arrays:")
print(f"  train_sequences : {train_sequences.shape}  dtype={train_sequences.dtype}")
print(f"  train_labels    : {train_labels.shape}  dtype={train_labels.dtype}")
print(f"  val_sequences   : {val_sequences.shape}  dtype={val_sequences.dtype}")
print(f"  val_labels      : {val_labels.shape}  dtype={val_labels.dtype}")
print(f"  test_sequences  : {test_sequences.shape}  dtype={test_sequences.dtype}")
print(f"  test_labels     : {test_labels.shape}  dtype={test_labels.dtype}")

Vocabulary size: 20,002

Loaded arrays:
  train_sequences : (40000, 500)  dtype=int32
  train_labels    : (40000,)  dtype=float32
  val_sequences   : (5000, 500)  dtype=int32
  val_labels      : (5000,)  dtype=float32
  test_sequences  : (5000, 500)  dtype=int32
  test_labels     : (5000,)  dtype=float32


## Section 2 — Dataset and DataLoaders

### Why wrap numpy arrays in a `Dataset`?

PyTorch's `DataLoader` expects a `Dataset` object — a class that implements `__len__` and
`__getitem__`. Wrapping our numpy arrays in a `Dataset` lets `DataLoader` handle:

- **Batching** — splitting the 40,000 training examples into mini-batches of 64
- **Shuffling** — randomising the order of batches each epoch so the model does not
  memorise the data ordering (only applied to the training set; validation and test are
  evaluated in a fixed order for reproducibility)
- **Tensor conversion** — converting numpy arrays to the correct `torch.Tensor` dtypes on the fly

### Why `batch_size=64`?

64 is a well-tested default for sequence models: small enough to fit in Colab's free GPU
memory (typically 15 GB), large enough that gradient estimates are stable and training
progresses quickly. If you encounter an out-of-memory error, reduce this to 32.

In [3]:
class IMDbDataset(Dataset):
    """Thin wrapper around pre-encoded IMDb sequences and labels."""

    def __init__(self, sequences: np.ndarray, labels: np.ndarray) -> None:
        # Convert to tensors once at construction time; avoids repeated conversion per batch
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return self.sequences[idx], self.labels[idx]


# ── Hyperparameters ───────────────────────────────────────────────────────
BATCH_SIZE = 64

# ── Wrap arrays in Dataset objects ───────────────────────────────────────
train_dataset = IMDbDataset(train_sequences, train_labels)
val_dataset   = IMDbDataset(val_sequences,   val_labels)
test_dataset  = IMDbDataset(test_sequences,  test_labels)

# ── Create DataLoaders ────────────────────────────────────────────────────
# shuffle=True only for training so every epoch sees examples in a different order
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches   : {len(train_loader):,}  ({len(train_dataset):,} examples)")
print(f"Val batches     : {len(val_loader):,}   ({len(val_dataset):,} examples)")
print(f"Test batches    : {len(test_loader):,}   ({len(test_dataset):,} examples)")

# Verify a single batch has the expected shapes
sample_seq, sample_lbl = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  sequences : {sample_seq.shape}  dtype={sample_seq.dtype}")
print(f"  labels    : {sample_lbl.shape}  dtype={sample_lbl.dtype}")

Train batches   : 625  (40,000 examples)
Val batches     : 79   (5,000 examples)
Test batches    : 79   (5,000 examples)

Sample batch shapes:
  sequences : torch.Size([64, 500])  dtype=torch.int64
  labels    : torch.Size([64])  dtype=torch.float32


## Section 3 — EarlyStopping and Helper Functions

### Why early stopping?

A common failure mode when training neural networks is **overfitting**: the model learns to
memorise the training data rather than generalise to new reviews. As overfitting worsens,
training loss continues to fall while validation loss *increases* — the two curves diverge.

Early stopping monitors validation loss after each epoch. If it fails to improve for
`patience=3` consecutive epochs, training stops and the best-performing checkpoint
(lowest validation loss) is used for evaluation. This prevents the model from wasting
compute on epochs that only hurt generalisation.

### Why `patience=3`?

A patience of 3 gives the model enough room to escape temporary plateaus (where validation
loss stalls for an epoch or two before improving again), while not waiting so long that we
train deep into an overfit regime.

### `train_epoch` and `eval_epoch`

Separating training and evaluation into dedicated functions keeps the main training loop
clean and makes each function independently testable. The key difference:

- `train_epoch`: calls `model.train()` (enables dropout), runs backprop and `optimizer.step()`
- `eval_epoch`: calls `model.eval()` (disables dropout), wraps the loop in `torch.no_grad()`
  to skip gradient computation (faster and uses less memory)

In [ ]:
class EarlyStopping:
    """Saves the best checkpoint and signals when validation loss has stopped improving."""

    def __init__(self, patience: int = 3, path: str = "checkpoints/best_model.pt") -> None:
        self.patience  = patience
        self.path      = path
        self.best_loss = float("inf")
        self.counter   = 0

    def __call__(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            os.makedirs(os.path.dirname(self.path), exist_ok=True)
            torch.save(model.state_dict(), self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for sequences, labels in loader:
        sequences, labels = sequences.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(sequences).squeeze(1)
        loss   = criterion(output, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += ((output >= 0.0).float() == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            output = model(sequences).squeeze(1)
            loss   = criterion(output, labels)
            total_loss += loss.item() * len(labels)
            correct    += ((output >= 0.0).float() == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


def run_training(config: dict, tag: str):
    print(f"\n{'='*60}")
    print(f"  {tag}")
    print(f"  num_layers={config['num_layers']}  hidden_size={config['hidden_size']}  "
          f"lr={config['learning_rate']}  patience={config['patience']}")
    print(f"{'='*60}")

    model = SentimentLSTM(
        vocab_size      = config["vocab_size"],
        embedding_dim   = config["embedding_dim"],
        hidden_size     = config["hidden_size"],
        num_layers      = config["num_layers"],
        dropout         = config["dropout"],
        use_batch_norm  = config["use_batch_norm"],
    ).to(device)

    optimizer      = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                      weight_decay=config.get("weight_decay", 0.0))
    criterion      = nn.BCEWithLogitsLoss()
    early_stopping = EarlyStopping(patience=config["patience"],
                                   path=config["checkpoint_path"])

    history = {"train_loss": [], "val_loss": [], "train_accuracy": [], "val_accuracy": []}

    for epoch in range(1, config["max_epochs"] + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        print(f"Epoch {epoch:2d}/{config['max_epochs']}  "
              f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        if early_stopping(val_loss, model):
            print(f"Early stopping triggered at epoch {epoch} "
                  f"(no improvement for {config['patience']} consecutive epochs).")
            print(f"Best val_loss: {early_stopping.best_loss:.4f}")
            break

    print(f"\nCheckpoint saved to: {config['checkpoint_path']}")
    return history


print("Helper classes and functions defined.")

## Section 4 — Plotting Helper

A single reusable function to draw the four training curves on one figure.
Plotting both loss and accuracy together on the same figure makes it easy to spot:

- **Overfitting**: train loss keeps falling, val loss diverges upward
- **Underfitting**: both losses remain high and plateau early
- **Healthy convergence**: both losses fall together and stabilise

In [5]:
def plot_history(history: dict, title: str) -> None:
    """Plot training and validation loss/accuracy curves."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # Loss subplot
    ax1.plot(epochs, history["train_loss"], "b-o", label="Train Loss")
    ax1.plot(epochs, history["val_loss"],   "r-o", label="Val Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("BCE Loss")
    ax1.set_title("Loss Curves")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy subplot
    ax2.plot(epochs, [a * 100 for a in history["train_accuracy"]], "b-o", label="Train Accuracy")
    ax2.plot(epochs, [a * 100 for a in history["val_accuracy"]],   "r-o", label="Val Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy Curves")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


print("Plotting helper defined.")

Plotting helper defined.


## Section 5 — Model Configuration and Training

### Architecture

| Component | Detail |
|---|---|
| Embedding | 20 002 tokens → 128-dim vectors, PAD frozen |
| LSTM | 3 stacked layers, hidden size 256, inter-layer dropout 0.4 |
| Batch Normalisation | Applied to the final hidden state before the classifier |
| Dropout | 0.4 applied after batch norm |
| Linear | 256 → 1 scalar logit |

### Why 3 layers and hidden size 256?

A single layer with 64 units is too shallow to capture the long-range compositional patterns that distinguish IMDb reviews (e.g., negations like "not bad at all").
Stacking three LSTM layers allows each layer to learn increasingly abstract representations — local word patterns at layer 1, phrase-level structure at layer 2, and discourse-level sentiment at layer 3.
Increasing the hidden size to 256 gives each layer enough capacity to retain relevant context across 500 tokens.

### Why batch normalisation?

Batch normalisation normalises the final hidden state across each mini-batch, reducing internal covariate shift.
This stabilises training, allows a slightly higher learning rate, and acts as a regulariser that complements dropout.

### Regularisation strategy

With more parameters comes higher overfitting risk.
Three measures keep the model in check:
- **Dropout 0.4** between LSTM layers and before the classifier
- **Batch normalisation** on the classifier input
- **Weight decay 1e-4** in Adam (L2 penalty on all weights)

### Early stopping

`patience=5` gives the model five non-improving epochs before stopping — more generous than the previous `patience=3` because the larger model may need a few extra epochs to settle after a temporary plateau.

In [ ]:
import pandas as pd

model_config = {
    "vocab_size"      : VOCAB_SIZE,
    "embedding_dim"   : 128,
    "hidden_size"     : 256,
    "num_layers"      : 3,
    "dropout"         : 0.4,
    "use_batch_norm"  : True,
    "learning_rate"   : 1e-3,
    "weight_decay"    : 1e-4,
    "max_epochs"      : 20,
    "patience"        : 5,
    "checkpoint_path" : "checkpoints/best_model.pt",
}

model_history = run_training(model_config, tag="3-Layer LSTM + BatchNorm (hidden=256)")

## Section 6 — Results

### Training History

In [ ]:
def history_table(history: dict) -> pd.DataFrame:
    epochs = list(range(1, len(history["train_loss"]) + 1))
    return pd.DataFrame({
        "Epoch"         : epochs,
        "Train Loss"    : [f"{v:.4f}" for v in history["train_loss"]],
        "Val Loss"      : [f"{v:.4f}" for v in history["val_loss"]],
        "Train Acc (%)" : [f"{v*100:.2f}" for v in history["train_accuracy"]],
        "Val Acc (%)"   : [f"{v*100:.2f}" for v in history["val_accuracy"]],
    })

history_table(model_history)

### Training Curves

In [ ]:
plot_history(model_history, "3-Layer LSTM + BatchNorm (hidden=256)")

### Interpretation

**Key things to observe in the curves:**

- **Loss convergence** — do training and validation loss fall together? If val loss starts rising while train loss keeps falling, the model is overfitting. The three regularisation measures (dropout, batch norm, weight decay) should delay this.
- **Train/val accuracy gap** — a gap larger than 5–10% indicates memorisation rather than generalisation.
- **Epoch count** — with patience=5, early stopping fires after five non-improving epochs. The best checkpoint (lowest val_loss) is what `05_evaluation.ipynb` will load.

The saved checkpoint at `checkpoints/best_model.pt` is ready for final test-set evaluation.